# Fourier-Native AMC on RadioML 2018.01A

This notebook is a thin experiment driver for the Fourier-native complex spectral model in `radioml/`. On Apple Silicon, the main reproducible entry point is `scripts/train_fourier_amc_mlx.py`.

## Setup

Install dependencies first if needed:

```bash
pip install -r requirements.txt
```

In [ ]:
import importlib.util
import os
import sys
from pathlib import Path

print(f'Notebook Python: {sys.executable}')
print('If imports are missing, install into this exact Python with:')
print(f'{sys.executable} -m pip install -r requirements.txt')

KAGGLE_DATASET = 'pinxau1000/radioml2018'
HDF5_PATTERNS = ['*GOLD*1024*.hdf5', '*RadioML*.hdf5', '*RML*.hdf5', '*.hdf5', '*.h5']

def kagglehub_cache_roots():
    owner, dataset = KAGGLE_DATASET.split('/')
    base = Path.home() / '.cache' / 'kagglehub' / 'datasets' / owner / dataset
    roots = []
    versions = base / 'versions'
    if versions.exists():
        roots.extend(sorted([p for p in versions.iterdir() if p.is_dir()], key=lambda p: p.stat().st_mtime, reverse=True))
    roots.append(base)
    return roots

def patch_kagglesdk_if_needed():
    try:
        import kagglesdk.kaggle_env as kaggle_env
    except Exception:
        return
    if not hasattr(kaggle_env, 'get_web_endpoint') and hasattr(kaggle_env, 'get_endpoint'):
        kaggle_env.get_web_endpoint = kaggle_env.get_endpoint
        print('Applied kagglehub/kagglesdk compatibility patch for this notebook session.')

def download_with_kagglehub():
    spec = importlib.util.find_spec('kagglehub')
    if spec is None:
        print('kagglehub is not visible to this notebook kernel.')
        print(f'Kernel executable: {sys.executable}')
        print(f'Install into this kernel with: {sys.executable} -m pip install kagglehub')
        return None

    patch_kagglesdk_if_needed()

    try:
        import kagglehub
    except Exception as exc:
        print(f'kagglehub was found at {spec.origin}, but importing it failed: {type(exc).__name__}: {exc}')
        return None

    path = Path(kagglehub.dataset_download(KAGGLE_DATASET))
    print(f'Path to dataset files: {path}')
    return path

def preview_files(root, limit=20):
    if root is None or not root.exists():
        return
    files = sorted(p for p in root.rglob('*') if p.is_file())[:limit]
    print('First downloaded/search files:')
    for file in files:
        print('  ', file)

def find_hdf5_under(root):
    if root is None or not root.exists():
        return None
    for pattern in HDF5_PATTERNS:
        matches = sorted([p for p in root.rglob(pattern) if p.is_file() and p.stat().st_size > 0])
        if matches:
            return matches[0]
    return None

def find_first_existing_hdf5(search_roots, label):
    for root in search_roots:
        match = find_hdf5_under(root)
        if match is not None:
            print(f'Found existing dataset in {label}; skipping Kaggle download.')
            return match
    return None

def find_radioml_file():
    env_path = os.environ.get('RADIOML_DATA')
    if env_path and Path(env_path).exists():
        path = Path(env_path)
        match = path if path.is_file() and path.stat().st_size > 0 else find_hdf5_under(path)
        if match is not None:
            print('Using RADIOML_DATA; skipping Kaggle download.')
            return match

    local_roots = [Path('data'), Path.home() / 'Downloads', Path.home() / 'Documents', Path.home() / 'Desktop']
    match = find_first_existing_hdf5(local_roots, 'local folders')
    if match is not None:
        return match

    match = find_first_existing_hdf5(kagglehub_cache_roots(), 'KaggleHub cache')
    if match is not None:
        return match

    download_root = download_with_kagglehub()
    match = find_hdf5_under(download_root)
    if match is None:
        preview_files(download_root)
    return match

DATA_PATH = find_radioml_file()
RUN_DIR = Path('runs/fourier_complex_mlx')

if DATA_PATH is None:
    print('RadioML HDF5 file not found yet. Put it in ./data/ or set os.environ["RADIOML_DATA"] to its path.')
else:
    print(f'Using dataset: {DATA_PATH}')

## Model Shape Check

This verifies the forward path on synthetic I/Q samples. It does not require the RadioML dataset.

In [ ]:
import mlx.core as mx
from radioml.mlx_model import FourierComplexAMCMLX

model = FourierComplexAMCMLX(num_classes=24, width=32, depth=2)
dummy_iq = mx.random.normal(shape=(4, 1024, 2))
logits = model(dummy_iq)
mx.eval(logits)

logits.shape

## Smoke Train

Use a tiny run first to catch data-shape and environment issues.

In [ ]:
import subprocess
import sys

if DATA_PATH is None or not DATA_PATH.exists():
    raise FileNotFoundError('Set DATA_PATH to your RadioML 2018.01A HDF5 file before training.')

subprocess.run([
    sys.executable, 'scripts/train_fourier_amc_mlx.py',
    '--data', str(DATA_PATH),
    '--epochs', '1',
    '--batch-size', '256',
    '--max-train', '4096',
    '--max-val', '1024',
    '--max-test', '1024',
    '--cache-data', 'ram',
    '--output-dir', str(RUN_DIR / 'smoke'),
], check=True)

## Fast Debug Run

Use this to check that validation rises above chance. It is not the final benchmark.

In [ ]:
if DATA_PATH is None or not DATA_PATH.exists():
    raise FileNotFoundError('Set DATA_PATH to your RadioML 2018.01A HDF5 file before training.')

subprocess.run([
    sys.executable, 'scripts/train_fourier_amc_mlx.py',
    '--data', str(DATA_PATH),
    '--epochs', '8',
    '--batch-size', '512',
    '--width', '32',
    '--depth', '4',
    '--dropout', '0.1',
    '--keep-bins', '512',
    '--max-train', '102400',
    '--max-val', '20480',
    '--max-test', '20480',
    '--steps-per-epoch', '200',
    '--val-steps', '40',
    '--cache-data', 'ram',
    '--output-dir', str(RUN_DIR / 'debug_fast'),
], check=True)

## Low-SNR Weighted Run

Run this after the fast debug run learns. It tests whether explicitly weighting `-20..0 dB` improves the target metric.

In [ ]:
if DATA_PATH is None or not DATA_PATH.exists():
    raise FileNotFoundError('Set DATA_PATH to your RadioML 2018.01A HDF5 file before training.')

subprocess.run([
    sys.executable, 'scripts/train_fourier_amc_mlx.py',
    '--data', str(DATA_PATH),
    '--epochs', '8',
    '--batch-size', '512',
    '--width', '32',
    '--depth', '4',
    '--dropout', '0.1',
    '--keep-bins', '512',
    '--low-snr-loss-weight', '2.0',
    '--max-train', '102400',
    '--max-val', '20480',
    '--max-test', '20480',
    '--steps-per-epoch', '200',
    '--val-steps', '40',
    '--cache-data', 'ram',
    '--output-dir', str(RUN_DIR / 'debug_low_snr_weighted'),
], check=True)

## Validation Sweep

Run all remaining trials. Completed trials are skipped and `sweep_results.csv` is rebuilt cumulatively, so this can be resumed across days.

In [ ]:
if DATA_PATH is None or not DATA_PATH.exists():
    raise FileNotFoundError('Set DATA_PATH to your RadioML 2018.01A HDF5 file before training.')

subprocess.run([
    sys.executable, 'scripts/sweep_fourier_amc_mlx.py',
    '--data', str(DATA_PATH),
    '--epochs', '1',
    '--batch-size', '512',
    '--max-train', '102400',
    '--max-val', '20480',
    '--cache-data', 'ram',
    '--skip-completed',
    '--output-dir', str(RUN_DIR / 'sweep'),
], check=True)

## 5-Epoch Validation Sweep

Use this after the 1-epoch sweep to compare learning trends more seriously. It writes to a separate folder so shallow and longer sweep results do not mix. Trials that reached 5 epochs are skipped; interrupted trials resume from `latest.safetensors`.

In [ ]:
if DATA_PATH is None or not DATA_PATH.exists():
    raise FileNotFoundError('Set DATA_PATH to your RadioML 2018.01A HDF5 file before training.')

subprocess.run([
    sys.executable, 'scripts/sweep_fourier_amc_mlx.py',
    '--data', str(DATA_PATH),
    '--epochs', '5',
    '--batch-size', '512',
    '--max-train', '102400',
    '--max-val', '20480',
    '--cache-data', 'ram',
    '--skip-completed',
    '--output-dir', str(RUN_DIR / 'sweep_5epoch'),
], check=True)

## Final Report Run

Use this only after selecting a config. It collects train accuracy and runs final test evaluation.

In [ ]:
if DATA_PATH is None or not DATA_PATH.exists():
    raise FileNotFoundError('Set DATA_PATH to your RadioML 2018.01A HDF5 file before training.')

subprocess.run([
    sys.executable, 'scripts/train_fourier_amc_mlx.py',
    '--data', str(DATA_PATH),
    '--epochs', '40',
    '--batch-size', '512',
    '--width', '64',
    '--depth', '6',
    '--dropout', '0.1',
    '--train-metrics', 'full',
    '--output-dir', str(RUN_DIR / 'final_report'),
], check=True)

## Inspect Run

In [ ]:
import json
import pandas as pd

RUN_NAME = 'debug_fast'
metrics_path = RUN_DIR / RUN_NAME / 'metrics.json'
if not metrics_path.exists():
    metrics_path = RUN_DIR / RUN_NAME / 'metrics_partial.json'
per_snr_path = RUN_DIR / RUN_NAME / 'per_snr_accuracy.csv'

metrics = json.loads(metrics_path.read_text())
history = pd.DataFrame([
    {
        'epoch': row['epoch'],
        'train_loss': row['train']['loss'],
        'train_acc': row['train'].get('accuracy'),
        'val_loss': row['val']['loss'],
        'val_acc': row['val']['accuracy'],
        'val_low_snr_acc': row['val']['low_snr_accuracy'],
    }
    for row in metrics['history']
])
per_snr = pd.read_csv(per_snr_path) if per_snr_path.exists() else None

display(history)
display(per_snr if per_snr is not None else 'per_snr_accuracy.csv is created after final test evaluation')
metrics.get('test')